In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
import sqlite3

In [3]:
con = sqlite3.connect(r"D:/Password_Strength_Prediction_using_NLP-main/Datasets/password_data.sqlite")

In [4]:
data = pd.read_sql_query("SELECT * FROM Users", con)

DatabaseError: Execution failed on sql 'SELECT * FROM Users': no such table: Users

In [ ]:
data.shape

In [ ]:
data.head(4)

In [ ]:
data.columns

In [ ]:
data.drop(["index"], axis=1, inplace=True)

In [ ]:
data.head(4)

In [ ]:
data.duplicated().sum()

In [ ]:
data.isnull().any()

In [ ]:
data.isnull().any().sum()

In [ ]:
data.dtypes

In [ ]:
#Check if any strength is -ve
data["strength"]

In [ ]:
data["strength"].unique()

In [ ]:
# Semantic analysis

In [ ]:
data.columns

In [ ]:
data["password"]

In [ ]:
type(data["password"][0])

In [ ]:
# How many password holds only numeric characters?
data["password"].str.isnumeric()

In [ ]:
data[data["password"].str.isnumeric()]

In [ ]:
data[data["password"].str.isnumeric()].shape

In [ ]:
# How many password hold upper case characters?
data[data["password"].str.isupper()].shape

In [ ]:
# How many password hold alphabetic characters?
data[data["password"].str.isalpha()].shape

In [ ]:
# How many password hold alpha-numeric characters?
data[data["password"].str.isalnum()].shape

In [ ]:
# How many password hold title characters?
data[data["password"].str.istitle()]

In [ ]:
data["password"]

In [ ]:
import string

In [ ]:
string.punctuation

In [ ]:
def find_semantics(row):
    for char in row:
        if char in string.punctuation:
            return 1
        else:
            pass

In [ ]:
data["password"].apply(find_semantics)==1

In [ ]:
data[data["password"].apply(find_semantics)==1]

In [ ]:
# Feature Engineering

In [ ]:
data["password"][0]

In [ ]:
len(data["password"][0])

In [ ]:
data["length"] = data["password"].str.len()

In [ ]:
password = "Sam04"

In [ ]:
[char for char in password if char.islower()]


In [ ]:
len([char for char in password if char.islower()])

In [ ]:
len([char for char in password if char.islower()])/len(password)

In [ ]:
def freq_lowercase(row):
    return len([char for char in row if char.islower()])/len(row)

In [ ]:
def freq_uppercase(row):
    return len([char for char in row if char.isupper()])/len(row)

In [ ]:
def freq_numerical_case(row):
    return len([char for char in row if char.isdigit()])/len(row)

In [ ]:
data["lowercase_freq"] = np.round(data["password"].apply(freq_lowercase), 3)

data["uppercase_freq"] = np.round(data["password"].apply(freq_uppercase), 3)

data["digit_freq"] = np.round(data["password"].apply(freq_numerical_case), 3)

In [ ]:
data.head(3)

In [ ]:
def freq_special_case(row):
    special_chars = []
    for char in row:
        if not char.isalpha() and not char.isdigit():
            special_chars.append(char)
    return len(special_chars)

In [ ]:
data["special_char_freq"] = np.round(data["password"].apply(freq_special_case), 3)

In [ ]:
data.head(5)

In [ ]:
data["special_char_freq"] = data["special_char_freq"]/data["length"]

In [ ]:
data.head(5)

In [ ]:
# Descriptive Statistics

In [ ]:
data.columns

In [ ]:
data[['length', 'strength']].groupby(['strength']).agg((["min", "max", "mean", "median"]))

In [ ]:
cols = ['length', 'lowercase_freq', 'uppercase_freq',
       'digit_freq', 'special_char_freq']

for col in cols:
    print(col)
    print(data[[col, 'strength']].groupby(['strength']).agg((["min", "max", "mean", "median"])))
    print('\n') 

In [ ]:
data.columns

In [ ]:
fig, ((ax1, ax2), (ax3, ax4), (ax5, ax6)) = plt.subplots(3, 2, figsize=(15,7))

sns.boxplot(x="strength", y='length', hue="strength", ax=ax1, data=data)
sns.boxplot(x="strength", y='lowercase_freq', hue="strength", ax=ax2, data=data)
sns.boxplot(x="strength", y='uppercase_freq', hue="strength", ax=ax3, data=data)
sns.boxplot(x="strength", y='digit_freq', hue="strength", ax=ax4, data=data)
sns.boxplot(x="strength", y='special_char_freq', hue="strength", ax=ax5, data=data)

plt.subplots_adjust(hspace=0.6)

In [ ]:
def get_dist(data, feature):

    plt.figure(figsize = (10, 8))
    plt.subplot(1,2,1)

    sns.violinplot(x='strength', y=feature,data=data)

    plt.subplot(1,2,2)

    sns.kdeplot(data[data['strength']==0][feature], color="red", label="0")
    sns.kdeplot(data[data['strength']==1][feature], color="blue", label="1")
    sns.kdeplot(data[data['strength']==2][feature], color="orange", label="2")
    plt.legend()
    plt.show()
    

In [ ]:
get_dist(data, "length")

In [ ]:
data.columns

In [ ]:
get_dist(data, "lowercase_freq")

In [ ]:
get_dist(data, "uppercase_freq")

In [ ]:
get_dist(data, "digit_freq")

In [ ]:
get_dist(data, "special_char_freq")

In [ ]:
data

In [ ]:
dataframe = data.sample(frac=1)

In [ ]:
dataframe

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
vectorizer = TfidfVectorizer(analyzer="char")

In [ ]:
x = list(dataframe["password"])

In [ ]:
X = vectorizer.fit_transform(x)

In [ ]:
X.shape

In [ ]:
dataframe["password"].shape

In [ ]:
X

In [ ]:
X.toarray()

In [ ]:
len(vectorizer.get_feature_names_out())

In [ ]:
df2 =pd.DataFrame(X.toarray(), columns=vectorizer.get_feature_names_out())

In [ ]:
df2

In [ ]:
dataframe.columns

In [ ]:
df2["length"] = dataframe["length"]
df2["lowercase_freq"] = dataframe["lowercase_freq"]

In [ ]:
df2

In [ ]:
from sklearn.model_selection import train_test_split
y = dataframe['strength']
X_train, X_test, y_train, y_test = train_test_split(df2, y, test_size=0.20)

In [ ]:
X_train.shape

In [ ]:
y_train.shape

In [ ]:
from sklearn.linear_model import LogisticRegression
clf = LogisticRegression(multi_class='multinomial')
clf.fit(X_train, y_train)

In [ ]:
import pickle

# Save the trained model
with open('model.pkl', 'wb') as f:
    pickle.dump(clf, f)

# If you have a vectorizer (like TfidfVectorizer), save it too:
# with open('vectorizer.pkl', 'wb') as f:
#     pickle.dump(vectorizer, f)

In [ ]:
y_pred = clf.predict(X_test)

In [ ]:
y_pred

In [ ]:
from collections import Counter
Counter(y_pred)

In [ ]:
password = "%@123abcd"

In [ ]:
sample_array = np.array([password])

In [ ]:
sample_matrix = vectorizer.transform(sample_array)
sample_matrix.toarray()

In [ ]:
sample_matrix.toarray().shape

In [ ]:
len(password)

In [ ]:
[char for char in password if char.islower()]

In [ ]:
len([char for char in password if char.islower()])/len(password)

In [ ]:
def predict():
    password = input("Enter a password: ")
    sample_array = np.array([password])
    sample_matrix = vectorizer.transform(sample_array)

    length_pass = len(password)
    length_normalized_lowercase = len([char for char in password if char.islower()])/len(password )

    # Create a new array with the additional features
    extra_features = np.array([[length_pass, length_normalized_lowercase]])

    # Concatenate the sample matrix with the extra features
    new_matrix = np.concatenate((sample_matrix.toarray(), extra_features), axis=1)

    result = clf.predict(new_matrix)

    if result == 0:
        return "Password is weak"
    elif result == 1:
        return "Password is normal"
    else:
        return "Password is strong"

In [ ]:
import warnings
warnings.filterwarnings("ignore")

predict()

In [ ]:
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report

accuracy_score(y_test, y_pred)

In [ ]:
confusion_matrix(y_test, y_pred)

In [ ]:
print(classification_report(y_test, y_pred))

In [ ]:
import pickle

# Save the vectorizer
with open('vectorizer.pkl', 'wb') as f:
    pickle.dump(vectorizer, f)